# NASA Space Apps EONET and ASF

Match EONET events to NISAR scene footprints from ASF, classify daily coverage, and prepare tables for BigQuery.

[Original Google Colab notebook](https://colab.research.google.com/drive/1CCGvYsKdbFVgnj44Gbqq_nJaFxv5oOBM?usp=sharing)

**Contents**

- [01 Set Up the Notebook](#setup)
  - [01.1 Install Dependencies](#install)
  - [01.2 Import Libraries and Set Defaults](#defaults)
  - [01.3 Configure BigQuery](#bigquery)
- [02 Extract and Prepare EONET Events](#eonet)
  - [02.1 Extract EONET Events](#extract-eonet)
  - [02.2 Assign Countries to Event Geometries](#assign-countries)
  - [02.3 Prepare EONET Records](#prepare-eonet)
- [03 Extract and Prepare ASF Scenes](#asf)
  - [03.1 Load Raw ASF Records in Batches](#load-asf)
  - [03.2 Prepare Scene Records and Thumbnails](#prepare-asf)
  - [03.3 Validate Scene Footprints](#validate-asf)
- [04 Match Events and Scenes](#matching)
- [05 Build Dimensions and the Event Hazard Bridge](#dimensions)
- [06 Publish and Verify BigQuery Tables](#publish)
  - [06.1 Publish Core Tables](#publish-core)
  - [06.2 Sync Reference Tables from Google Sheets](#sync-sheets)
  - [06.3 Verify Published Tables](#verify-tables)

<a id="setup"></a>

## 01 Set Up the Notebook

Run the setup cells before starting the pipeline. The notebook can write to BigQuery. Review the destination and publishing settings first.

<a id="install"></a>

### 01.1 Install Dependencies

In [ ]:
%pip -q install -U asf_search "geopandas>=1.0" "shapely>=2.0" google-cloud-bigquery openpyxl

<a id="defaults"></a>

### 01.2 Import Libraries and Set Defaults

In [ ]:
import json
import re
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import asf_search as asf
import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import shapely
from IPython.display import Markdown, display
from shapely.geometry import shape

DATA_INICIO = "2025-08-01"
hoje_sp = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
DATA_FIM = (hoje_sp - timedelta(days=1)).isoformat()

MODO_TESTE_ASF = True
TESTAR_IMAGEM = False
PUBLICAR_BIGQUERY = False
SUBSTITUIR_TABELAS = False
PROJETO_GCP = ""  # Google Cloud project ID
DATASET_BQ = ""   # Existing BigQuery dataset
LOCAL_BQ = "US"  # Match the dataset location

# print(f"Date range: {DATA_INICIO} a {DATA_FIM}")

<a id="bigquery"></a>

### 01.3 Configure BigQuery

In [ ]:
# 01.3 BigQuery connection

PROJETO_GCP = ""
DATASET_BQ = ""
LOCAL_BQ = "US"
PUBLICAR_BIGQUERY = False

In [ ]:
# 01.3 BigQuery publishing helpers

import json
import re

import numpy as np
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from shapely.geometry.base import BaseGeometry

# An empty set creates missing tables and skips existing ones.
# To replace a single table, specify its name, for example {"db_asf"}
ATUALIZAR_TABELAS = set()

destino = f"{PROJETO_GCP}.{DATASET_BQ}"

bq = None
tabelas_existentes = set()
if PROJETO_GCP and DATASET_BQ:
    auth.authenticate_user()
    bq = bigquery.Client(project=PROJETO_GCP, location=LOCAL_BQ)
    tabelas_existentes = {t.table_id for t in bq.list_tables(destino)}

tipos_bq = {
    "data_utc": "DATE",
    "data_captura_utc": "DATE",
    "data_evento_utc": "TIMESTAMP",
    "captura_utc": "TIMESTAMP",
    "latitude": "FLOAT64",
    "longitude": "FLOAT64",
    "pct_pixels_alterados": "FLOAT64",
    "qtd_cenas_associadas": "INT64",
    "indicio_mudanca": "BOOL",
    "asf_amostra": "BOOL",
}

def valor_bq(valor, tipo):
    if isinstance(valor, BaseGeometry):
        return valor.wkt
    if isinstance(valor, (list, tuple, dict, np.ndarray)):
        dados = valor.tolist() if isinstance(valor, np.ndarray) else valor
        return json.dumps(dados, ensure_ascii=False, default=str)
    if pd.isna(valor):
        return None
    if tipo in {"DATE", "TIMESTAMP"}:
        return valor.isoformat()
    return {"INT64": int, "FLOAT64": float, "BOOL": bool}.get(tipo, str)(valor)

def carregar_tabela(nome, df):
    existe = nome in tabelas_existentes
    atualizar = nome in ATUALIZAR_TABELAS

    if existe and not atualizar:
        print(f"Already published · {nome}")
        return

    if df is None or df.empty:
        print(f"Pending · {nome} · DataFrame is empty or unavailable")
        return

    acao = "Replace" if existe else "Create"

    if not PUBLICAR_BIGQUERY:
        print(f"Preview · {acao} {nome} · {len(df):,} rows")
        return

    if bq is None:
        raise ValueError("Configure PROJETO_GCP and DATASET_BQ before publishing.")

    colunas = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in df.columns]
    if len({c.lower() for c in colunas}) != len(colunas):
        raise ValueError(f"{nome}: duplicate column names after normalization.")

    tabela_id = f"{destino}.{nome}"
    modo = "WRITE_TRUNCATE" if existe else "WRITE_EMPTY"
    print(f"Preparing · {nome} · {len(df):,} rows", flush=True)

    if nome == "db_asf_raw":
        from tempfile import TemporaryFile

        raw = df.copy()
        raw.columns = colunas

        for coluna in raw.columns:
            valores = raw[coluna]
            if any(isinstance(v, (list, tuple, dict, np.ndarray)) for v in valores):
                valores = valores.map(lambda v: valor_bq(v, "STRING"))
            raw[coluna] = valores.astype("string")

        config = bigquery.LoadJobConfig(
            schema=[bigquery.SchemaField(c, "STRING") for c in colunas],
            source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
            write_disposition=modo,
        )

        with TemporaryFile(mode="w+b") as arquivo:
            arquivo.write(
                raw.to_json(orient="records", lines=True, force_ascii=False)
                .encode("utf-8")
            )
            arquivo.seek(0)

            print(f"Uploading · {nome}", flush=True)
            job = bq.load_table_from_file(arquivo, tabela_id, job_config=config)
            print(f"BigQuery job submitted · {job.job_id}", flush=True)
            job.result()
    else:
        tipos = [tipos_bq.get(c, "STRING") for c in colunas]
        schema = [bigquery.SchemaField(c, t) for c, t in zip(colunas, tipos)]
        registros = [
            {c: valor_bq(v, t) for c, v, t in zip(colunas, linha, tipos)}
            for linha in df.itertuples(index=False, name=None)
        ]

        config = bigquery.LoadJobConfig(schema=schema, write_disposition=modo)
        print(f"Uploading · {nome}", flush=True)
        job = bq.load_table_from_json(registros, tabela_id, job_config=config)
        print(f"BigQuery job submitted · {job.job_id}", flush=True)
        job.result()

    tabelas_existentes.add(nome)
    print(f"Published · {nome} · {len(df):,} rows")

# Compatibility wrapper for the optional Drive import.
def publicar_tabela(cliente, nome, dados):
    carregar_tabela(nome, dados)


<a id="eonet"></a>

## 02 Extract and Prepare EONET Events

<a id="extract-eonet"></a>

### 02.1 Extract EONET Events

The extraction cell sets the analysis window. It ends on the previous calendar day in São Paulo, and the geospatial match uses UTC dates.

In [ ]:
# 02.1 Extract EONET events

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import geopandas as gpd
import pandas as pd
import requests
import shapely
from IPython.display import display
from shapely.geometry import shape

# Shared date range for EONET, ASF, and the date dimension.

hoje_sp = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
DATA_INICIO = (pd.Timestamp(hoje_sp) - pd.DateOffset(months=6)).date().isoformat()
DATA_FIM = (hoje_sp - timedelta(days=1)).isoformat()

print(f"Data range: {DATA_INICIO} a {DATA_FIM} UTC")

PAISES_EXCLUIDOS = {"CHN"}

url_fronteiras = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
    "master/geojson/ne_50m_admin_0_countries.geojson"
)

fronteiras_base = gpd.read_file(url_fronteiras).to_crs("EPSG:4326")

mascara_exclusao = (
    fronteiras_base["ISO_A3"].isin(PAISES_EXCLUIDOS)
    | fronteiras_base["ADMIN"].eq("China")
)

if not mascara_exclusao.any():
    raise ValueError("The boundary for China was not found. Extraction stopped.")

area_excluida = shapely.union_all(
    fronteiras_base.loc[mascara_exclusao].geometry.make_valid().array
)

if area_excluida.is_empty:
    raise ValueError("The exclusion geometry is empty. Extraction stopped.")

shapely.prepare(area_excluida)

def geometria_permitida(geo):
    if not isinstance(geo, dict) or not geo.get("coordinates"):
        return False

    geometria = shapely.make_valid(shape(geo))
    return not geometria.is_empty and not area_excluida.intersects(geometria)

print(f"Querying EONET: {DATA_INICIO} a {DATA_FIM} UTC", flush=True)

resposta = requests.get(
    "https://eonet.gsfc.nasa.gov/api/v3/events",
    params={"status": "all", "start": DATA_INICIO, "end": DATA_FIM},
    timeout=120,
)
resposta.raise_for_status()

eventos_recebidos = resposta.json()["events"]
eventos_permitidos = [
    evento
    for evento in eventos_recebidos
    if evento.get("geometry")
    and all(geometria_permitida(geo) for geo in evento["geometry"])
]

df_eonet_raw = pd.DataFrame(eventos_permitidos)

print(f"Events retained: {len(df_eonet_raw):,} of {len(eventos_recebidos):,}")
if df_eonet_raw.empty:
    raise ValueError("No EONET events remain after filtering.")

display(df_eonet_raw.head())

<a id="assign-countries"></a>

### 02.2 Assign Countries to Event Geometries

Country codes and names come from the boundary data. A country name can be available even when its ISO code is missing. [Natural Earth boundaries](https://github.com/nvkelso/natural-earth-vector/blob/master/geojson/ne_50m_admin_0_countries.geojson)

In [ ]:
url_fronteiras = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
    "master/geojson/ne_50m_admin_0_countries.geojson"
)
fronteiras = (
    gpd.read_file(url_fronteiras)[["ISO_A3", "ADMIN", "geometry"]]
    .rename(columns={"ISO_A3": "codigo_pais", "ADMIN": "pais"})
    .to_crs("EPSG:4326")
    .reset_index(drop=True)
)
codigo_valido = fronteiras["codigo_pais"].str.fullmatch(r"[A-Z]{3}", na=False)
fronteiras.loc[~codigo_valido, "codigo_pais"] = None
fronteiras["geometry"] = fronteiras.geometry.make_valid()

linhas = []
for evento in df_eonet_raw.to_dict("records"):
    categorias = evento.get("categories") or []
    for geo in evento.get("geometry") or []:
        if geo.get("type") not in {"Point", "Polygon", "MultiPolygon"} or not geo.get("coordinates"):
            continue
        geometria = shapely.make_valid(shape(geo))
        if geometria.is_empty:
            continue
        rows.append({
            "id_evento_eonet": evento["id"],
            "titulo_evento": evento.get("title"),
            "desastre": ", ".join(
                str(c.get("title") or c["id"]) for c in categorias
            ) or None,
            "data_original": geo.get("date"),
            "geometria_evento": geometria,
        })

geometrias = gpd.GeoDataFrame(
    pd.DataFrame(linhas, columns=[
        "id_evento_eonet", "titulo_evento", "desastre",
        "data_original", "geometria_evento",
    ]),
    geometry="geometria_evento", crs="EPSG:4326",
)
eonet_com_pais = gpd.sjoin(geometrias, fronteiras, how="left", predicate="intersects")
eonet_com_pais = eonet_com_pais.reset_index(drop=True)
com_pais = eonet_com_pais["index_right"].notna()
eonet_com_pais.loc[com_pais, "geometria_evento"] = shapely.intersection(
    eonet_com_pais.loc[com_pais].geometry.array,
    eonet_com_pais.loc[com_pais, "index_right"].map(fronteiras.geometry).to_numpy(),
)
eonet_com_pais["pais"] = eonet_com_pais["pais"].fillna("Sem país")

# print(f"Geometries assigned to a country: {com_pais.sum():,}")

<a id="prepare-eonet"></a>

### 02.3 Prepare EONET Records

In [ ]:
df_eonet = eonet_com_pais.drop(columns="index_right").copy()
df_eonet["data_evento_utc"] = pd.to_datetime(
    df_eonet["data_original"], format="ISO8601", utc=True, errors="coerce",
)
inicio_utc = pd.Timestamp(DATA_INICIO, tz="UTC")
fim_exclusivo_utc = pd.Timestamp(DATA_FIM, tz="UTC") + pd.Timedelta(days=1)
df_eonet = df_eonet.loc[
    df_eonet["data_evento_utc"].ge(inicio_utc)
    & df_eonet["data_evento_utc"].lt(fim_exclusivo_utc)
].reset_index(drop=True)
df_eonet["data_utc"] = df_eonet["data_evento_utc"].dt.date

# Coordenadas do marcador para o mapa do Power BI
marcadores = df_eonet["geometria_evento"].representative_point()
df_eonet["latitude"] = marcadores.y
df_eonet["longitude"] = marcadores.x


# print(f"Geometry records in the date range: {len(df_eonet):,}")

display(df_eonet.drop(columns="geometria_evento").head())

<a id="asf"></a>

## 03 Extract and Prepare ASF Scenes

<a id="load-asf"></a>

### 03.1 Load Raw ASF Records in Batches

This cell appends to the existing, non-partitioned BigQuery raw table. Review its destination and start date before running it. A day with rows already present is skipped.

In [ ]:
# 03.1 Load raw ASF records in seven-day batches

import json
from tempfile import TemporaryFile

import asf_search as asf
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

INICIO_LOTE_ASF = "2026-07-03"
DIAS_POR_LOTE = 7

if PUBLICAR_BIGQUERY and PROJETO_GCP and DATASET_BQ:
    PROJETO = PROJETO_GCP
    ALVO = f"{PROJETO}.{DATASET_BQ}.db_asf_raw"

    if "geometria_permitida" not in globals() or "DATA_FIM" not in globals():
        raise RuntimeError("Run section 02.1 before loading ASF records.")

    primeiro_dia = pd.Timestamp(INICIO_LOTE_ASF, tz="UTC")
    fim_periodo = pd.Timestamp(DATA_FIM, tz="UTC")

    if primeiro_dia > fim_periodo:
        raise ValueError("The batch start date is after DATA_FIM.")

    auth.authenticate_user()
    cliente = bigquery.Client(project=PROJETO, location=LOCAL_BQ)
    tabela = cliente.get_table(ALVO)

    if tabela.time_partitioning is not None:
        raise ValueError("The load requires the existing non-partitioned db_asf_raw table.")

    campos = {
        "properties_fileID": "fileID",
        "properties_startTime": "startTime",
        "properties_url": "url",
        "properties_browse": "browse",
        "properties_polarization": "polarization",
        "properties_mainBandPolarization": "mainBandPolarization",
        "properties_sideBandPolarization": "sideBandPolarization",
    }

    config_carga = bigquery.LoadJobConfig(
        schema=tabela.schema,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition="WRITE_APPEND",
    )
    config_consulta = bigquery.QueryJobConfig(use_query_cache=False)

    def como_texto(valor):
        if valor is None or isinstance(valor, str):
            return valor
        return json.dumps(valor, ensure_ascii=False)

    def contar_dia(dia):
        sql = f"""
            SELECT COUNT(*) AS rows
            FROM `{ALVO}`
            WHERE data_captura_utc = DATE '{dia}'
        """
        return next(iter(cliente.query(sql, job_config=config_consulta).result())).linhas

    def carregar_dia(data):
        dia = data.date().isoformat()
        existentes = contar_dia(dia)

        if existentes:
            print(f"Already loaded · {dia} · {existentes:,} rows", flush=True)
            return

        ids_enviados = set()

        with TemporaryFile(mode="w+b") as arquivo:
            for hora in range(0, 24, 6):
                inicio = data + pd.Timedelta(hours=hora)
                fim = inicio + pd.Timedelta(hours=6)
                fim_api = fim - pd.Timedelta(microseconds=1)

                print(f"{dia} · ASF · {inicio:%H:%M}–{fim:%H:%M} UTC", flush=True)

                for pagina in asf.search_generator(
                    dataset="NISAR",
                    processingLevel="GCOV",
                    start=inicio.isoformat(),
                    end=fim_api.isoformat(),
                ):
                    for produto in pagina:
                        cena = produto.geojson()
                        props = cena["properties"]
                        cena_id = props.get("fileID")
                        captura = pd.to_datetime(
                            props.get("startTime"), utc=True, errors="coerce"
                        )

                        if not cena_id or cena_id in ids_enviados:
                            continue
                        if pd.isna(captura) or not (inicio <= captura < fim):
                            continue
                        if not geometria_permitida(cena.get("geometry")):
                            continue

                        registro = {
                            "data_captura_utc": dia,
                            **{
                                nome: como_texto(props.get(campo))
                                for nome, campo in campos.items()
                            },
                            "registro_geojson": json.dumps(cena, ensure_ascii=False),
                        }
                        arquivo.write(
                            (json.dumps(registro, ensure_ascii=False) + "\n").encode("utf-8")
                        )
                        ids_enviados.add(cena_id)

            if not ids_enviados:
                print(f"{dia} · no products after filtering", flush=True)
                return

            arquivo.seek(0)
            job = cliente.load_table_from_file(arquivo, ALVO, job_config=config_carga)
            print(f"{dia} · BigQuery job: {job.job_id}", flush=True)
            job.result()

        gravadas = contar_dia(dia)
        if gravadas != len(ids_enviados):
            raise RuntimeError(
                f"{dia}: expected {len(ids_enviados):,} rows and found "
                f"{gravadas:,}. Job: {job.job_id}"
            )

        print(f"VERIFIED IN BIGQUERY · {dia} · {gravadas:,} rows", flush=True)

    inicio_lote = primeiro_dia

    while inicio_lote <= fim_periodo:
        fim_lote = min(
            inicio_lote + pd.Timedelta(days=DIAS_POR_LOTE - 1),
            fim_periodo,
        )

        print(f"\nBatch · {inicio_lote:%Y-%m-%d} a {fim_lote:%Y-%m-%d} UTC", flush=True)

        for dia_lote in pd.date_range(inicio_lote, fim_lote):
            carregar_dia(dia_lote)

        inicio_lote = fim_lote + pd.Timedelta(days=1)

    print(f"Processed through {fim_periodo:%Y-%m-%d} · {ALVO}")
else:
    print('ASF load skipped. Configure BigQuery and enable publishing to run this step.')


<a id="prepare-asf"></a>

### 03.2 Prepare Scene Records and Thumbnails

This step expects an in-memory `df_asf_raw` with normalized GeoJSON fields. The batch cell above writes to BigQuery and does not create this DataFrame.

In [ ]:
campos_asf = {
    "properties.fileID": "id_cena_asf",
    "properties.startTime": "captura_original",
    "geometry.type": "tipo_geometria",
    "geometry.coordinates": "coordenadas_cena",
    "properties.browse": "urls_previa",
    "properties.url": "url_produto_radar",
}
df_asf = df_asf_raw.reindex(columns=list(campos_asf)).rename(columns=campos_asf)
df_asf["captura_utc"] = pd.to_datetime(
    df_asf["captura_original"], format="ISO8601", utc=True, errors="coerce",
)
df_asf["data_captura_utc"] = df_asf["captura_utc"].dt.date
cenas_sem_id = int(df_asf["id_cena_asf"].isna().sum())
ids_repetidos = int(df_asf["id_cena_asf"].dropna().duplicated().sum())
df_asf = df_asf.dropna(subset=["id_cena_asf"]).drop_duplicates("id_cena_asf").reset_index(drop=True)

# Select one thumbnail per scene.
links = df_asf[["id_cena_asf", "urls_previa"]].explode("urls_previa")
eh_miniatura = links["urls_previa"].astype("string").str.endswith(
    "_thumbnail.png", na=False
)

miniaturas = links.loc[eh_miniatura].drop_duplicates("id_cena_asf")
miniatura_por_cena = miniaturas.set_index("id_cena_asf")["urls_previa"]

df_asf["url_miniatura"] = df_asf["id_cena_asf"].map(miniatura_por_cena)


display(df_asf[[
    "id_cena_asf",
    "captura_utc",
    "data_captura_utc",
    "tipo_geometria",
    "coordenadas_cena",
    "url_miniatura",
]].head(10))

# print(f"Missing IDs: {cenas_sem_id:,} | Duplicate IDs: {ids_repetidos:,}")

**Optional test — Check scene thumbnails**

Set `TESTAR_IMAGEM` to `True` to request the first thumbnail URL and print its HTTP status and content type.

In [ ]:
for cena in miniaturas.head(2).itertuples(index=False):
    display(Markdown(f"- `{cena.id_cena_asf}` · [Open thumbnail]({cena.urls_previa})"))

if miniaturas.empty:
    print("No thumbnails were found in the loaded records.")
elif TESTAR_IMAGEM:
    url_teste = miniaturas.iloc[0]["urls_previa"]
    with requests.get(url_teste, timeout=30, stream=True) as teste:
        print(f"HTTP {teste.status_code} | {teste.headers.get('Content-Type', '')}")

**Optional enrichment — Attach existing image comparisons**

The notebook imports comparison results from `df_comparacao` if present. Each result must have a unique later-scene identifier.

In [ ]:
df_asf["url_miniatura"] = df_asf["id_cena_asf"].map(
    miniaturas.set_index("id_cena_asf")["urls_previa"]
).astype("string")

campos_comparacao = {
    "id_cena_antes": "id_cena_referencia",
    "pct_pixels_alterados": "pct_pixels_alterados",
    "indicio_mudanca": "indicio_mudanca",
}
for coluna, tipo in zip(campos_comparacao.values(), ["string", "Float64", "boolean"]):
    df_asf[coluna] = pd.Series(pd.NA, index=df_asf.index, dtype=tipo)

if "df_comparacao" in globals():
    comparacoes = df_comparacao.dropna(subset=["id_cena_depois"]).set_index(
        "id_cena_depois", verify_integrity=True,
    )
    for origem, destino in campos_comparacao.items():
        df_asf[destino] = df_asf["id_cena_asf"].map(comparacoes[origem]).astype(
            df_asf[destino].dtype
        )

display(df_asf[["id_cena_asf", "url_miniatura", "indicio_mudanca"]].head(10))

<a id="validate-asf"></a>

### 03.3 Validate Scene Footprints

In [ ]:
def geometria_cena(tipo, coords):
    if tipo in {"Polygon", "MultiPolygon"} and isinstance(coords, (list, tuple)) and coords:
        geom = shapely.make_valid(shape({"type": tipo, "coordinates": coords}))
        if not geom.is_empty and geom.geom_type in {"Polygon", "MultiPolygon"}:
            return geom
    return None

df_asf["geometria_cena"] = [
    geometria_cena(tipo, coords)
    for tipo, coords in zip(df_asf["tipo_geometria"], df_asf["coordenadas_cena"])
]
assert df_asf["id_cena_asf"].is_unique
print(f"Scenes with no date: {df_asf['captura_utc'].isna().sum():,}")
print(f"Scenes with no usable footprint: {df_asf['geometria_cena'].isna().sum():,}")
print(f"Scenes with a thumbnail: {df_asf['url_miniatura'].notna().sum():,}")

<a id="matching"></a>

## 04 Match Events and Scenes

The join uses footprint intersection and the same UTC date. Daily coverage uses the union of matched footprints. A no-match result reflects the ASF records analyzed. Set `asf_e_amostra` to indicate whether coverage uses an ASF sample. Spatial operations are planar in EPSG:4326. Review antimeridian cases before publishing global results.

In [ ]:
chaves = ["id_evento_eonet", "pais", "data_utc"]
eventos_dia = df_eonet.dissolve(
    by=chaves,
    aggfunc={"titulo_evento": "first", "desastre": "first", "codigo_pais": "first"},
    as_index=False,
)

cenas_validas = df_asf.dropna(subset=["data_captura_utc", "geometria_cena"])
cenas_geo = gpd.GeoDataFrame(
    cenas_validas.rename(columns={"data_captura_utc": "data_utc"}),
    geometry="geometria_cena", crs="EPSG:4326",
)
pares = gpd.sjoin(
    eventos_dia,
    cenas_geo[["data_utc", "id_cena_asf", "captura_utc", "url_miniatura", "geometria_cena"]],
    how="inner", predicate="intersects", on_attribute="data_utc",
).reset_index(drop=True)

pares["geometria_cena"] = pares["index_right"].map(cenas_geo.geometry)
pares["cobertura_cena"] = [
    "Sim" if cena.covers(evento) else "Parcialmente"
    for cena, evento in zip(pares["geometria_cena"], pares.geometry)
]
df_asf_incident = pd.DataFrame(pares[[
    *chaves, "codigo_pais", "desastre", "id_cena_asf", "captura_utc",
    "url_miniatura", "cobertura_cena",
]]).copy()

resumos = []
for chave, grupo in pares.groupby(chaves, sort=False):
    uniao = shapely.union_all(grupo["geometria_cena"].to_numpy())
    resumos.append({
        **dict(zip(chaves, chave)),
        "cobertura_dia": "Sim" if uniao.covers(grupo.geometry.iloc[0]) else "Parcialmente",
        "qtd_cenas_associadas": grupo["id_cena_asf"].nunique(),
    })

resumo = pd.DataFrame(resumos, columns=[*chaves, "cobertura_dia", "qtd_cenas_associadas"])
df_eonet_monitor = pd.DataFrame(eventos_dia.drop(columns="geometria_evento")).merge(
    resumo, on=chaves, how="left", validate="one_to_one",
)
df_eonet_monitor["cobertura_dia"] = df_eonet_monitor["cobertura_dia"].fillna("Não")
df_eonet_monitor["qtd_cenas_associadas"] = (
    df_eonet_monitor["qtd_cenas_associadas"].fillna(0).astype("int64")
)
df_eonet_monitor["asf_amostra"] = asf_e_amostra



print(f"Events by country and day: {len(df_eonet_monitor):,} | Event-scene pairs: {len(df_asf_incident):,}")

display(df_eonet_monitor.head())
display(df_asf_incident.head())

<a id="dimensions"></a>

## 05 Build Dimensions and the Event Hazard Bridge

The bridge retains each category ID assigned to an event.

In [ ]:
# 05 Build dimensions and the event hazard bridge

# Included countries and territories.
# Keep the complete country boundaries for spatial matching.
dim_pais = (
    fronteiras.loc[
        ~fronteiras["codigo_pais"].isin(PAISES_EXCLUIDOS),
        ["codigo_pais", "pais"],
    ]
    .dropna(subset=["codigo_pais"])
    .drop_duplicates(subset=["codigo_pais"])
    .sort_values("pais")
    .reset_index(drop=True)
)

# Full EONET category catalog.
resposta_categorias = requests.get(
    "https://eonet.gsfc.nasa.gov/api/v3/categories",
    timeout=30,
)
resposta_categorias.raise_for_status()

dim_desastres = (
    pd.DataFrame(resposta_categorias.json()["categories"])
    .rename(columns={"id": "id_categoria_eonet", "title": "fenomeno"})
    [["id_categoria_eonet", "fenomeno", "description"]]
    .sort_values("fenomeno")
    .reset_index(drop=True)
)

dim_desastres["id_categoria_eonet"] = (
    dim_desastres["id_categoria_eonet"].astype("string")
)

# One row per retained event and assigned category.
eventos_mantidos = df_eonet_raw.loc[
    df_eonet_raw["id"].isin(df_eonet["id_evento_eonet"]),
    ["id", "categories"],
]

ponte_evento_desastre = pd.DataFrame(
    [
        (evento_id, categoria["id"])
        for evento_id, categorias in eventos_mantidos.itertuples(
            index=False, name=None
        )
        for categoria in (categorias or [])
        if categoria.get("id") is not None
    ],
    columns=["id_evento_eonet", "id_categoria_eonet"],
).drop_duplicates(ignore_index=True)

ponte_evento_desastre["id_categoria_eonet"] = (
    ponte_evento_desastre["id_categoria_eonet"].astype("string")
)

# One row per day in the shared period, through yesterday.
dim_data = pd.DataFrame({
    "data_utc": pd.date_range(start=DATA_INICIO, end=DATA_FIM).date
})

print(f"Countries and territories retained: {len(dim_pais):,}")
print(f"Hazards in the EONET catalog: {len(dim_desastres):,}")
print(
    "Hazards present in prepared events: "
    f"{ponte_evento_desastre['id_categoria_eonet'].nunique():,}"
)
print(f"Dates: {len(dim_data):,} ({DATA_INICIO} a {DATA_FIM})")
print(f"Excluded codes remaining in dim_pais: "
      f"{dim_pais['codigo_pais'].isin(PAISES_EXCLUIDOS).sum()}")

display(dim_desastres)

<a id="publish"></a>

## 06 Publish and Verify BigQuery Tables

The publishing helper serializes nested values as JSON and Shapely geometries as WKT. It normalizes column names and assigns explicit BigQuery date and timestamp types. Use `ATUALIZAR_TABELAS` to select existing tables for replacement.

<a id="publish-core"></a>

### 06.1 Publish Core Tables

The coverage and scene-detail loads require `inicio_asf` to identify the start of the ASF period.

In [ ]:
# 06.1 Publish core tables


oficiais = {
    "db_eonet_raw": "df_eonet_raw",
    "db_eonet": "df_eonet",
    "db_eonet_monitor": "df_eonet_monitor",
    "db_asf_raw": "df_asf_raw",
    "db_asf": "df_asf",
    "db_asf_incident": "df_asf_incident",
}

for nome, variavel in oficiais.items():
    df = globals().get(variavel)
    precisa_carregar = nome not in tabelas_existentes or nome in ATUALIZAR_TABELAS

    if precisa_carregar and df is not None:
        if nome in {"db_eonet_monitor", "db_asf_incident"}:
            if "inicio_asf" not in globals() or "DATA_FIM" not in globals():
                print(f"Pending · {nome} · ASF date range unavailable")
                continue

            inicio = pd.Timestamp(inicio_asf).date()
            fim = pd.Timestamp(DATA_FIM).date()
            df = df.loc[df["data_utc"].between(inicio, fim)]

    carregar_tabela(nome, df)

<a id="sync-sheets"></a>

### 06.2 Sync Reference Tables from Google Sheets

In [ ]:
# Sync reference tables from Google Sheets

import pandas as pd

base = ""  # Published Google Sheets base URL

abas_apoio = {
    "db_hazard_guidance": ("", "id_categoria_eonet"),
    "db_emergency_contacts": ("", "codigo_pais"),
}

if base and all(gid for gid, _ in abas_apoio.values()):
    for nome, (gid, coluna_chave) in abas_apoio.items():
        url_csv = f"{base}?gid={gid}&single=true&output=csv"
        dados = pd.read_csv(url_csv, dtype="string")
        dados.columns = dados.columns.str.strip()
        dados = dados.replace(r"^\s*$", pd.NA, regex=True).dropna(how="all")

        if nome == "db_emergency_contacts" and "codigo_pais" not in dados.columns:
            dados = dados.rename(columns={"id_pais": "codigo_pais"})

        if coluna_chave not in dados.columns:
            raise ValueError(
                f"{nome}: expected column {coluna_chave!r}. "
                f"Headers found: {dados.columns.tolist()}"
            )

        if nome == "db_emergency_contacts":
            codigos = dados["codigo_pais"].str.strip().str.upper()

            if not codigos.str.fullmatch(r"[A-Z]{3}", na=False).all():
                raise ValueError(
                    "In the contacts sheet, fill id_pais with "
                    "three-letter country codes, such as BRA."
                )

            dados["codigo_pais"] = codigos
            dados = dados.loc[~codigos.isin(PAISES_EXCLUIDOS)].copy()

        print(f"Google Sheets · {nome} · {len(dados):,} rows")
        carregar_tabela(nome, dados)
else:
    print('Reference sync skipped. Supply the published Sheets URL and both tab IDs.')


**Optional alternative — Import reference tables from Drive**

In [ ]:
# Optional reference-table import from Drive
ARQUIVO_DB_FENOM = ""
ARQUIVO_DB_DEFENCE = ""

if not ARQUIVO_DB_FENOM or not ARQUIVO_DB_DEFENCE:
    print("Enter both full paths when the reference files are available.")
else:
    from google.colab import drive
    drive.mount("/content/drive")

    arquivos = {"db_fenom": ARQUIVO_DB_FENOM, "db_defence": ARQUIVO_DB_DEFENCE}
    apoio = {}
    for nome, arquivo in arquivos.items():
        extensao = Path(arquivo).suffix.lower()
        if extensao == ".csv":
            apoio[nome] = pd.read_csv(arquivo, dtype="string", sep=None, engine="python")
        elif extensao == ".xlsx":
            apoio[nome] = pd.read_excel(arquivo, dtype="string")
        else:
            raise ValueError(f"{arquivo}: export as CSV or XLSX.")
        display(apoio[nome].head())

    db_fenom = apoio["db_fenom"]
    db_defence = apoio["db_defence"]
    if PUBLICAR_BIGQUERY:
        bq = cliente_bigquery()
        for nome, dados in apoio.items():
            publicar_tabela(bq, nome, dados)

**Optional alternative — Import reference tables with authenticated Google Sheets access**

In [ ]:
# Sync reference tables from Google Sheets

import gspread
from google.auth import default

planilhas_apoio = {
    "db_fenomen": {"url": "", "aba": ""},
    "db_defence": {"url": "", "aba": ""},
}

if any(config["url"] for config in planilhas_apoio.values()):
    credenciais, _ = default()  # Reuse the authentication from section 01.3.
    sheets = gspread.authorize(credenciais)

    for nome, config in planilhas_apoio.items():
        if not config["url"]:
            print(f"Pending · {nome} · enter the spreadsheet link")
            continue

        planilha = sheets.open_by_url(config["url"])
        aba = planilha.worksheet(config["aba"]) if config["aba"] else planilha.sheet1
        valores = aba.get_all_values()

        if len(valores) < 2:
            print(f"Pending · {nome} · the worksheet needs a header and data")
            continue

        dados = pd.DataFrame(
            valores[1:],
            columns=[coluna.strip() for coluna in valores[0]],
        )
        dados = dados.replace(r"^\s*$", pd.NA, regex=True).dropna(how="all")

        if nome == "db_defence":
            if "codigo_pais" not in dados.columns:
                raise ValueError("Use the codigo_pais header in db_defence.")

            codigos = dados["codigo_pais"].astype("string").str.strip().str.upper()
            if not codigos.str.fullmatch(r"[A-Z]{3}", na=False).all():
                raise ValueError("In db_defence, use three-letter country codes in codigo_pais, such as BRA.")

            dados["codigo_pais"] = codigos
            dados = dados.loc[
                ~dados["codigo_pais"].isin(PAISES_EXCLUIDOS)
            ].copy()

        print(f"Google Sheets · {nome} · {len(dados):,} rows to load")
        carregar_tabela(nome, dados)
else:
    print("Enter the links for db_fenomen and db_defence.")

<a id="verify-tables"></a>

### 06.3 Verify Published Tables

In [ ]:
# 06.3 Verify published tables
import pandas as pd

destino = f"{PROJETO_GCP}.{DATASET_BQ}"

esperadas = {
    "Core tables": [
        "db_eonet_raw", "db_eonet", "db_eonet_monitor",
        "db_asf_raw", "db_asf", "db_asf_incident",
    ],
    "Dimensions and bridge": [
        "dim_country", "dim_date", "dim_hazard", "bridge_event_hazard",
    ],
    "Reference tables": ["db_fenomen", "db_defence"],
}

if bq is not None:
    publicadas = {t.table_id: t.reference for t in bq.list_tables(destino)}
    checagem = []

    for grupo, nomes in esperadas.items():
        for nome in nomes:
            referencia = publicadas.get(nome)
            rows = bq.get_table(referencia).num_rows if referencia else None

            checagem.append({
                "Group": grupo,
                "Table": nome,
                "Status": "OK" if referencia else "Pending",
                "Rows": rows,
            })

    print(f"Dataset: {destino}")
    display(pd.DataFrame(checagem).astype({"Rows": "Int64"}))
else:
    print('Verification skipped. Configure BigQuery first.')
